Structured Streaming engine takes care of running query incrementally and continuously as new data arrives into the system

Structured Streaming is a stream processing framework built on top of Spark SQL engine.Structure streaming is same as writting a batch computation on static data

Structured Streaming ensures end-to-end, exactly-once processing as well as fault-tolerance through checkpointing and write-ahead logs.

The main idea behind Structured Streaming is to treat a stream of data as a table to which data is continuously appended


In stream processing systems there are effectively two relevant times for each event:
- The time at which it actually occurred (event time),
- The time that it was processed or reached the stream processing system (processing time)

 Note : The order of the series of events in the processing system does not guarantee ordering in event time
 we must acknowledge that any number of things can happen to the events on the way from the source of the information to our stream processing system
 We need to operate on event time and look at the overall stream with reference to this information contained in the data
 rather than on when it arrives in the system


**1. Stream (Input Stream)**

`df = spark.readStream.format("kafka").load()`

**2. Trigger**
To control when data is output to our sink, we set a trigger
A ProcessingTime trigger runs on a fixed schedule. If we set it to 1 minute, Spark will output results exactly at 12:00, 12:01, 12:02, and so on

a) Processing Time Trigger - 	Runs a batch at a fixed interval.

	`.trigger(processingTime="10 seconds") --> “Process whatever data has arrived  every 10 seconds”`

```
activityCounts.writeStream\
  .trigger(processingTime='5 seconds')\
  .format("console")\
  .outputMode("complete")\
  .start()
```



b). Once Trigger- Processes all available data once, then stops


`  .trigger(once=True)`

```
activityCounts.writeStream\
  .trigger(once=True)\
  .format("console")\
  .outputMode("complete")\
  .start()
```



c) Available Now (Databricks / newer Spark)


  `.trigger(availableNow=True) --> Better than once for large data volumes.`

**3. Watermark**

Watermark defines how late data is allowed to arrive.

	.withWatermark("event_time", "10 minutes") --> “I expect data to arrive at most 10 minutes late.
	Anything older than that can be dropped.”

**4. Window**

	Used for time-based aggregations.
	Example: 10-minute window
	from pyspark.sql.functions import window
	df.groupBy(
		window("event_time", "10 minutes")
	).count() -->Group events that happened in the same 10-minute interval


**5.Sliding Window**

- Overlapping windows

	window("event_time", "10 minutes", "1 minutes")

	Example

	Window: 10 minutes

	Slide: 1 minute
  
	You get updates every minute about the last 10 minutes

**6.Stateful Operation**



```

When performing a stateful operation, Spark stores the intermediate information in a state store
 Spark’s current state store implementation is an in-memory state store, it is made fault tolerant by storing intermediate
state to the checkpoint directory

from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StringType

schema = StructType().add("user", StringType())

df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .schema(schema) \
    .load()

user_counts = df.groupBy("user").count()
At first glance it looks simple, but in Structured Streaming this is STATEFUL.
State = memory of past data across micro-batches

Assume data arrives in micro-batches:
Batch 1
	Alice
	Bob
	Alice
	Result:
	Alice → 2
	Bob   → 1

Spark stores this internally as state:
	State Store:
	Alice = 2
	Bob   = 1
Batch 2
	Bob
	Alice
	Charlie

To compute the new result, Spark must do:
	Alice = 2 (old) + 1 = 3
	Bob   = 1 (old) + 1 = 2
	Charlie = 0 + 1 = 1


query = user_counts.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("checkpointLocation", "/tmp/checkpoints/user_counts") \
    .start()


 **Arbitrary Stateful Processing**
 Control over what state should be stored, how it is updated, and when it should be removed is called arbitrary (or custom) stateful processing

some examples:
we like to record information about user sessions on an ecommerce site. For
instance, we want to track what pages the users visit over the course of this session in order to provide recommendations in real time during their next session.
These sessions have completely arbitrary start and stop times that are unique to that user.

The company  like to report on errors in the web application but only if five
events occur during a user’s session
```



**7. Stateless Opeartion**


Stateless operations process each record independently. Spark does not remember previous data.

Each row is independent.

Examples:

		*  select
		*  map
		*  filter
No memory required.

**8. Output Mode**

Defines what gets written to the sink.

	a) Append- 	Only new rows are written.Ensures that each row is output once (and only once), assuming that we have a fault-tolerant sink.
	 Use when rows are final and never change
	.outputMode("append")

	b) Update - only changed rows are written.
	Update mode is similar to complete mode except that only the rows that are different from the previous write are written out to the sink.
	Use when rows may change
	.outputMode("update")

	c) Complete - Writes entire result table every batch.
	Complete mode will output the entire state of the result table to your output sink.
	Use when the entire result table must be output every time
	.outputMode("complete")

**9. Sink (Output)**

	Where the processed stream goes.

**10. Checkpointing**

Used for fault tolerance.

	.option("checkpointLocation", "/mnt/checkpoints/stream1")

# Sample code

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CSV_Read").getOrCreate()


to read schema -- we will read the schema from one file (that we know has a valid schema) and pass the
dataSchema object from our static DataFrame to our streaming DataFrame

In [ ]:
static = spark.read.json("/content/sample_data/Streamdata/*.json")
dataSchema = static.schema

In [ ]:
static = spark.read.json("/content/sample_data/Streamdata/*.json")
dataSchema = static.schema


StructType([StructField('Arrival_Time', LongType(), True), StructField('Creation_Time', LongType(), True), StructField('Device', StringType(), True), StructField('Index', LongType(), True), StructField('Model', StringType(), True), StructField('User', StringType(), True), StructField('gt', StringType(), True), StructField('x', DoubleType(), True), StructField('y', DoubleType(), True), StructField('z', DoubleType(), True)])

In [ ]:
from pprint import pp

pp(dataSchema)



StructType([StructField('Arrival_Time', LongType(), True), StructField('Creation_Time', LongType(), True), StructField('Device', StringType(), True), StructField('Index', LongType(), True), StructField('Model', StringType(), True), StructField('User', StringType(), True), StructField('gt', StringType(), True), StructField('x', DoubleType(), True), StructField('y', DoubleType(), True), StructField('z', DoubleType(), True)])




```

 Note: Structured Streaming does not let you perform schema inference without explicitly enabling it
 enable schema inference for this by setting the configuration spark.sql.streaming.schemaInference to true


streaming = spark\
	.readStream\
	.schema(dataSchema)\
	.option("maxFilesPerTrigger", 10)\
.json("/content/sample_data/Streamdata/*.json")

The schema looks like
streaming.printSchema()

root
|-- Arrival_Time: long (nullable = true)
|-- Creation_Time: long (nullable = true)
|-- Device: string (nullable = true)
|-- Index: long (nullable = true)
|-- Model: string (nullable = true)
|-- User: string (nullable = true)
|-- gt: string (nullable = true)
|-- x: double (nullable = true)
|-- y: double (nullable = true)
|-- z: double (nullable = true)

```

Note: streaming DataFrame creation and execution is lazy.
Streaming DataFrames don’t run immediately. You first define all your transformations and the stream only starts when you call an action like .start()

https://spark.apache.org/docs/latest/streaming/getting-started.html

https://spark.apache.org/docs/latest/streaming/index.html



*Let say here, Creation_Time is the Event time and Arrival_Time is the Processing time*

**we like to aggregate events in a window of 10 mins**



```
withEventTime = streaming\
			.selectExpr("*","cast(cast(Creation_Time as double)/1000000000 as timestamp) as event_time")

from pyspark.sql.functions import window, col
withEventTime.groupBy(window(col("event_time"), "10 minutes")).count()\
			.writeStream\
			.queryName("events_per_window")\
			.format("memory")\
			.outputMode("complete")\
			.start()

spark.sql("SELECT * FROM events_per_window").printSchema()
SELECT * FROM events_per_window
```



*As we have done aggregation based on 1 col only, we can do aggregation based on multi col too.

Events/per User in a span of 10 minutes*



```
from pyspark.sql.functions import window, col
withEventTime.groupBy(window(col("event_time"), "10 minutes"), "User").count()\
			.writeStream\
			.queryName("pyevents_per_window")\
			.format("memory")\
			.outputMode("complete")\
			.start()

      spark.sql("SELECT * FROM events_per_window").printSchema()
```






```
Sliding Window
--------------
A sliding window is just a time window that moves forward in small steps, producing overlapping windows.

let say Window = 10 min  and overlap =5 min

Window 1: [0 ----------- 10]
Window 2:       [5 ----------- 15]
Window 3:             [10 ----------- 20]
Window 4:                   [15 ----------- 25]
Window 5:                         [20 ----------- 30]
```





```
from pyspark.sql.functions import window, col

withEventTime.groupBy(window(col("event_time"), "10 minutes", "5 minutes"))\
			.count()\
			.writeStream\
			.queryName("pyevents_per_window")\
			.format("memory")\
			.outputMode("complete")\
			.start()


SELECT * FROM events_per_window
```



**Handling Late Data with Watermarks**


A Watermark is an amount of time following a given event or set of events after
which we do not expect to see any more data from that time.

we need to specify watermarks because if we did not, we’d need to keep all of our windows around forever, expecting them to be updated forever. This brings us to the core question when working with event-time: “how late do I expect to see data?” The answer to this question will be the watermark that we’ll configure for our data

```
if we know that we typically see data as produced downstream in minutes
but we have seen delays in events up to 30 minutes after they occur (perhaps the user lost cell phone connectivity), we’d specify the watermark in the following way:


from pyspark.sql.functions import window, col
withEventTime\
	.withWatermark("event_time", "30 minutes")\
	.groupBy(window(col("event_time"), "10 minutes", "5 minutes"))\
	.count()\
	.writeStream\
	.queryName("pyevents_per_window")\
	.format("memory")\
	.outputMode("complete")\
	.start()




 Now, Structured Streaming will wait until 30 minutes after the final
timestamp of this 10-minute rolling window before it finalizes the result of that window

Note :  if we do not specify how late we think we will see data, then Spark will maintain that data in memory forever.
Specifying a watermark allows it to free those objects from memory, allowing our stream to continue running for a longtime

```


**Dropping Duplicates in a Stream**

To de-duplicate data, Spark will maintain a number of user specified keys and ensure that duplicates are ignored
in this example -  duplicate events will have the same timestamp as well as user


from pyspark.sql.functions import expr

withEventTime\
    .withWatermark("event_time", "5 seconds")\
    .**dropDuplicates**(["User", "event_time"])\
    .groupBy("User")\
    .count()\
    .writeStream\
    .queryName("deduplicated")\
    .format("memory")\
    .outputMode("complete")\
    .start()


For Woking code refer : https://colab.research.google.com/drive/1gJWA3so-087itRxqMCVg2xWxmxLxdKBk#scrollTo=XC2ziPAzbiaH